# POC: LEAF

📖 対応記事: `親子対話：LEAF`🔗 [記事を読む](https://github.com/bonsai/sound-gen/blob/main/articles/%E8%A6%AA%E5%AD%90%E5%AF%BE%E8%A9%B1%EF%BC%9ALEAF.md)

In [ ]:
# @title Setup
!pip install -q torch numpy matplotlib

In [ ]:
# @title Demo
# LEAF: Learnable Audio Frontend
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Minimal LEAF-inspired frontend
class LeafFrontend(nn.Module):
  def __init__(self, n_filters=40, kernel_size=401):
    super().__init__()
    # Gabor filters (learnable)
    self.freq = nn.Parameter(torch.linspace(80, 7600, n_filters) / 16000)
    self.sigma = nn.Parameter(torch.ones(n_filters) * 0.1)
    self.kernel_size = kernel_size
  
  def forward(self, x):
    B, T = x.shape
    t = torch.arange(-(self.kernel_size//2), self.kernel_size//2 + 1, device=x.device).float()
    # Gabor filterbank
    kernels = []
    for f, s in zip(self.freq, self.sigma):
      gabor = torch.exp(-(t**2) / (2*s**2)) * torch.cos(2*np.pi*f*t)
      kernels.append(gabor)
    kernel = torch.stack(kernels).unsqueeze(1)  # [C, 1, K]
    return F.conv1d(x.unsqueeze(1), kernel, padding=self.kernel_size//2)

# Demo
model = LeafFrontend(20, 201)
audio = torch.randn(2, 16000)
feats = model(audio)
print(f"LEAF Frontend: {audio.shape} → {feats.shape}")
print(f"✅ LEAF replaces Mel filterbank with learnable Gabor filters")
print(f"   End-to-end: learns optimal time-frequency representation for task")

---
### 📝 まとめ
TODO

🔗 [記事を読む](https://github.com/bonsai/sound-gen/blob/main/articles/%E8%A6%AA%E5%AD%90%E5%AF%BE%E8%A9%B1%EF%BC%9ALEAF.md)